# 06 — Generator Cost Parameters

**Purpose:** Attach heat rates, fuel costs, and O&M costs to every operating generator,
producing a bus-indexed cost table for the Julia case builder.

**Inputs:**
- `data/processed/background files/power_plants.geojson` — EIA operating generators
- `data/processed/background files/bus_locations.geojson` — bus topology with plant aggregates
- `data/processed/background files/grid_network.graphml` — NetworkX graph for component analysis
- EIA API v2: `electricity/facility-fuel` (heat rates) and `electricity/electric-power-operational-data` (fuel costs)
- NREL ATB 2024: `https://oedi-data-lake.s3.amazonaws.com/ATB/electricity/csv/2024/v3.0.0/ATBe.csv`

**Outputs:**
- `data/processed/network_metadata.json` — island filter threshold and graph summary stats
- `data/processed/generators_with_costs.parquet` — per-generator cost table
- `data/processed/cost_coverage.csv` — coverage summary by technology

**Data resolution notes:**
- Heat rates: plant+fuel level annual average from Form EIA-923 (facility-fuel endpoint)
- Fuel costs: state+fuel level annual average from Form EIA-923 (electric-power-operational-data)
  — plant-level costs are not exposed by EIA API v2; state average is the finest available
- O&M: NREL ATB 2024 Moderate scenario; Nuclear O&M hardcoded (ATB 2024 omits nuclear O&M)

In [ ]:
import sys, json
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
from tqdm import tqdm
import utils

BG        = PROJECT_ROOT / "data" / "processed" / "background files"
PROCESSED = PROJECT_ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

print("Imports OK")

## Step 0 — Load and verify required files

In [ ]:
print("Loading files...")

plants = gpd.read_file(BG / "power_plants.geojson")
buses  = gpd.read_file(BG / "bus_locations.geojson")
G      = nx.read_graphml(BG / "grid_network.graphml")

# ── Verify required columns ───────────────────────────────────────────────────
req_plant_cols = ['plantid', 'generatorid', 'technology', 'energy-source-desc',
                  'energy_source_code', 'nameplate-capacity-mw', 'stateid']
missing = [c for c in req_plant_cols if c not in plants.columns]
assert not missing, f"Missing plant columns: {missing}"

req_bus_cols = ['bus_id', 'total_capacity_mw', 'dominant_fuel', 'degree']
missing = [c for c in req_bus_cols if c not in buses.columns]
assert not missing, f"Missing bus columns: {missing}"

sample_node = dict(list(G.nodes(data=True))[0][1])
assert 'bus_id' in sample_node,         "graph nodes missing bus_id"
assert 'total_capacity_mw' in sample_node or True, "OK — only plant-bearing nodes have capacity"

print(f"plants  : {plants.shape[0]:,} rows, {len(plants.columns)} cols")
print(f"buses   : {buses.shape[0]:,} rows  |  buses with plants: {(buses['plant_count']>0).sum():,}")
print(f"graph   : {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")
print(f"Total snapped MW in graph: {sum(d.get('total_capacity_mw',0) for _,d in G.nodes(data=True)):,.0f}")
print()
print("Plants columns:", plants.columns.tolist())
print("heat-rate present:", any('heat' in c.lower() for c in plants.columns))
print("capacity dtype:", plants['nameplate-capacity-mw'].dtype)

# Coerce capacity to float now; used throughout notebook
plants = plants.copy()
plants['capacity_mw'] = pd.to_numeric(plants['nameplate-capacity-mw'], errors='coerce').fillna(0.0)

## Step 1 — Capacity Retention Diagnostic

For each connected component, sum the total snapped capacity (MW).  Print MW
retained at node-count thresholds 1, 10, 50, 100 to determine the island
filter cutoff for the Julia case builder.

In [ ]:
print("─" * 60)
print("STEP 1 — Capacity retention diagnostic")
print("─" * 60)

comps = list(nx.connected_components(G))

comp_rows = []
for comp in comps:
    size = len(comp)
    mw   = sum(G.nodes[n].get('total_capacity_mw', 0) for n in comp)
    comp_rows.append({'n_nodes': size, 'total_mw': float(mw)})

comp_df   = pd.DataFrame(comp_rows).sort_values('n_nodes', ascending=False).reset_index(drop=True)
total_mw  = comp_df['total_mw'].sum()
giant_size = comp_df['n_nodes'].iloc[0]

print(f"  Total components        : {len(comp_df):,}")
print(f"  Giant component nodes   : {giant_size:,}")
print(f"  Total MW in graph       : {total_mw:,.1f}")
print()
print(f"  {'Threshold':>12}  {'#Comps':>8}  {'Retained MW':>12}  {'% Total':>8}")
print("  " + "-" * 48)

threshold_results = {}
for thresh in [1, 5, 10, 25, 50, 100, 500]:
    mask      = comp_df['n_nodes'] >= thresh
    ret_mw    = comp_df.loc[mask, 'total_mw'].sum()
    n_comps   = mask.sum()
    pct       = 100 * ret_mw / total_mw if total_mw > 0 else 0
    threshold_results[thresh] = {'n_comps': int(n_comps), 'retained_mw': ret_mw, 'pct': pct}
    print(f"  ≥{thresh:>11,}  {n_comps:>8,}  {ret_mw:>12,.1f}  {pct:>7.2f}%")

# Determine plateau: first threshold where retained% ≥ 95%
island_filter = None
for thresh in [1, 5, 10, 25, 50, 100, 500]:
    if threshold_results[thresh]['pct'] >= 95.0:
        island_filter = thresh
        break
if island_filter is None:
    island_filter = 50  # fallback

print()
print(f"  → island_filter_min_nodes = {island_filter}  "
      f"(first threshold retaining ≥95% of total MW)")

## Step 2 — Write network_metadata.json

In [ ]:
print("─" * 60)
print("STEP 2 — Write network_metadata.json")
print("─" * 60)

# Identify which nodes belong to components above the filter threshold
giant_comp_nodes = set(
    n for comp in comps if len(comp) >= island_filter for n in comp
)

metadata = {
    "total_buses"           : G.number_of_nodes(),
    "total_lines"           : G.number_of_edges(),
    "giant_component_size"  : int(giant_size),
    "n_components"          : len(comp_df),
    "grid_size_m"           : 500,
    "island_filter_min_nodes": int(island_filter),
    "pct_mw_retained_at_filter": round(threshold_results[island_filter]['pct'], 4),
    "total_snapped_mw"      : round(total_mw, 2),
    "data_vintage_year"     : 2024,
    "atb_scenario"          : "Moderate",
    "atb_version"           : "2024 v3.0.0",
    "notes": (
        "island_filter_min_nodes is the minimum component size for inclusion in the "
        "Julia power flow case. Buses in smaller components are flagged in_giant_component=false."
    )
}

meta_path = PROCESSED / "network_metadata.json"
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print("  Wrote:", meta_path)
print(json.dumps(metadata, indent=2))

## Step 3 — Re-snap Plants to Buses

The bus_locations GeoDataFrame has aggregate attributes only.  We need
per-generator bus assignments, so we redo the spatial snap here.

In [ ]:
print("─" * 60)
print("STEP 3 — Snap plants to buses")
print("─" * 60)

MAX_SNAP_M = 50_000  # 50 km

plants_proj = plants.to_crs(epsg=5070)
buses_proj  = buses[['bus_id', 'geometry']].to_crs(epsg=5070)

snapped = gpd.sjoin_nearest(
    plants_proj,
    buses_proj,
    how='left',
    max_distance=MAX_SNAP_M,
    distance_col='snap_dist_m',
)
# Drop duplicates from equidistant matches (keep closest)
snapped = (
    snapped
    .sort_values('snap_dist_m', na_position='last')
    .drop_duplicates(subset=['plantid', 'generatorid', 'period'], keep='first')
)

excluded = snapped[snapped['bus_id'].isna()]
included = snapped[snapped['bus_id'].notna()].copy()
included['bus_id'] = included['bus_id'].astype(int)

# Mark in_giant_component
# Convert graph node IDs (strings in graphml) to int for lookup
giant_comp_nodes_int = {int(n) for n in giant_comp_nodes}
included['in_giant_component'] = included['bus_id'].isin(giant_comp_nodes_int)

print(f"  Snapped  : {len(included):,}")
print(f"  Excluded : {len(excluded):,}  (> {MAX_SNAP_M/1000:.0f} km or no nearby bus)")
print(f"  In-giant : {included['in_giant_component'].sum():,}  "
      f"({100*included['in_giant_component'].mean():.1f}%)")

## Step 4 — Heat Rates from EIA facility-fuel

`electricity/facility-fuel` (Form EIA-923) provides plant+prime-mover level
annual fuel consumption (MMBtu) and net generation (MWh).  
Heat rate = consumption_mmbtu / generation_mwh.

Join key: `plantid` ↔ `plantCode`, `energy_source_code` ↔ `fuel2002`.
Non-thermal generators (wind, solar, hydro, storage) are left as null.

In [ ]:
print("─" * 60)
print("STEP 4 — Fetch heat rates (facility-fuel, annual 2024)")
print("─" * 60)

# Non-thermal fuels — no heat rate exists for these
NO_HEAT_RATE_FUELS = {
    'WAT', 'WND', 'SUN', 'GEO',    # renewables
    'ES', 'PH', 'FW',               # storage / pumped hydro / flywheel
    'WH', 'PUR',                    # waste heat / purchased steam
    'MWH',                          # electricity input (charging)
}

PAGE_SIZE = 5000
facility_records = []
offset = 0
MAX_RETRIES = 2

while True:
    params = {
        'frequency' : 'annual',
        'start'     : '2024',
        'end'       : '2024',
        'data[0]'   : 'generation',
        'data[1]'   : 'consumption-for-eg-btu',
        # No primeMover filter — fetch all prime movers, aggregate by plant+fuel in Python
        'sort[0][column]'    : 'plantCode',
        'sort[0][direction]' : 'asc',
        'sort[1][column]'    : 'fuel2002',
        'sort[1][direction]' : 'asc',
        'length' : PAGE_SIZE,
        'offset' : offset,
    }
    records = []
    for attempt in range(MAX_RETRIES):
        try:
            resp = utils.eia_get('electricity/facility-fuel/data', params)
            records = resp.get('response', {}).get('data', [])
            break
        except Exception as e:
            if attempt < MAX_RETRIES - 1:
                print(f"  Retry {attempt+1} after error: {e}")
            else:
                print(f"  Skipping offset {offset} after {MAX_RETRIES} failures: {e}")
    facility_records.extend(records)
    print(f"  Page offset {offset:>6,}: fetched {len(records):,} "
          f"(running total: {len(facility_records):,})")
    if len(records) < PAGE_SIZE:
        break
    offset += PAGE_SIZE

ff_df = pd.DataFrame(facility_records)
print(f"  Total facility-fuel rows (2024, all prime movers): {len(ff_df):,}")
print(f"  Columns: {ff_df.columns.tolist()}")
print(f"  Sample:")
print(ff_df.head(3).to_string())

In [ ]:
# ── Compute heat rates ────────────────────────────────────────────────────────
# NOTE: despite the field name 'consumption-for-eg-btu', units are MMBtu
# (confirmed from API units field: 'MMBtu').  heat_rate = MMBtu / MWh directly.

ff_df['generation_mwh']    = pd.to_numeric(ff_df['generation'],               errors='coerce')
ff_df['consumption_mmbtu'] = pd.to_numeric(ff_df['consumption-for-eg-btu'],   errors='coerce')

# Remove aggregate 'ALL' fuel rows and 'ALL' prime-mover rows
ff_fuel = ff_df[
    (ff_df['fuel2002']   != 'ALL') &
    (ff_df['primeMover'] != 'ALL')
].copy()

# Aggregate across prime movers → plant + fuel level total
ff_plant_fuel = (
    ff_fuel
    .groupby(['plantCode', 'fuel2002'], as_index=False)
    .agg(generation_mwh=('generation_mwh', 'sum'),
         consumption_mmbtu=('consumption_mmbtu', 'sum'))
)

# Compute heat rate; guard against zero or near-zero generation
MIN_GEN_MWH = 1.0
ff_plant_fuel['heat_rate_mmbtu_mwh'] = np.where(
    ff_plant_fuel['generation_mwh'] >= MIN_GEN_MWH,
    ff_plant_fuel['consumption_mmbtu'] / ff_plant_fuel['generation_mwh'],
    np.nan,
)

# Sanity filter: plausible heat rate range 3–50 MMBtu/MWh
mask_valid = (
    (ff_plant_fuel['heat_rate_mmbtu_mwh'] >= 3) &
    (ff_plant_fuel['heat_rate_mmbtu_mwh'] <= 50)
)
ff_valid = ff_plant_fuel[mask_valid].copy()
print(f"  Plant+fuel rows (after prime-mover aggregation): {len(ff_plant_fuel):,}")
print(f"  Valid heat rate rows (3–50 MMBtu/MWh): {len(ff_valid):,}")

# Build lookup: plantCode × fuel2002 → heat_rate
ff_valid['plantCode'] = ff_valid['plantCode'].astype(str)
hr_lookup = (
    ff_valid[['plantCode', 'fuel2002', 'heat_rate_mmbtu_mwh']]
    .rename(columns={'plantCode': 'plantid', 'fuel2002': 'energy_source_code'})
)
print(f"  Heat rate lookup entries: {len(hr_lookup):,}")
print(f"  Heat rate distribution:")
print(hr_lookup['heat_rate_mmbtu_mwh'].describe().to_string())

# ── Join to generators ────────────────────────────────────────────────────────
included['plantid_str'] = included['plantid'].astype(str)
included2 = included.merge(
    hr_lookup,
    left_on=['plantid_str', 'energy_source_code'],
    right_on=['plantid',    'energy_source_code'],
    how='left',
    suffixes=('', '_lookup'),
)
# Explicitly null for non-thermal fuels
included2.loc[
    included2['energy_source_code'].isin(NO_HEAT_RATE_FUELS),
    'heat_rate_mmbtu_mwh'
] = np.nan

n_with_hr  = included2['heat_rate_mmbtu_mwh'].notna().sum()
n_thermal  = (~included2['energy_source_code'].isin(NO_HEAT_RATE_FUELS)).sum()
print(f"\n  Generators with heat rate       : {n_with_hr:,}")
print(f"  Thermal generators (should have): {n_thermal:,}")
print(f"  Coverage among thermal gens     : {100*n_with_hr/n_thermal:.1f}%")

## Step 5 — Fuel Costs from EIA electric-power-operational-data

`electric-power-operational-data` exposes state+sector+fueltypeid averages from
Form EIA-923.  Field `cost-per-btu` is already in **$/MMBtu** (API label is
misleading; units string confirms "dollars per million Btu").

Plant-level costs are not available through EIA API v2.  We join on
state × mapped fuel type — the finest resolution available.

In [ ]:
print("─" * 60)
print("STEP 5 — Fetch fuel costs (electric-power-operational-data, annual 2024)")
print("─" * 60)

# Fuel types that actually have measurable cost (exclude renewables/nuclear handled separately)
# We pull all and filter later
cost_records = []
offset = 0

while True:
    resp = utils.eia_get('electricity/electric-power-operational-data/data', {
        'frequency' : 'annual',
        'start'     : '2024',
        'end'       : '2024',
        'data[0]'   : 'cost-per-btu',
        'facets[sectorid][0]': '98',  # all sectors combined
        'sort[0][column]'    : 'location',
        'sort[0][direction]' : 'asc',
        'length' : PAGE_SIZE,
        'offset' : offset,
    })
    records = resp.get('response', {}).get('data', [])
    cost_records.extend(records)
    print(f"  Page offset {offset:>6,}: fetched {len(records):,} "
          f"(running total: {len(cost_records):,})")
    if len(records) < PAGE_SIZE:
        break
    offset += PAGE_SIZE

# If sectorid=98 returned nothing, retry without sector filter
if not cost_records:
    print("  Retrying without sectorid filter...")
    resp = utils.eia_get('electricity/electric-power-operational-data/data', {
        'frequency' : 'annual',
        'start'     : '2024',
        'end'       : '2024',
        'data[0]'   : 'cost-per-btu',
        'length'    : PAGE_SIZE,
        'offset'    : 0,
    })
    cost_records = resp.get('response', {}).get('data', [])

cost_df_raw = pd.DataFrame(cost_records)
print(f"  Total rows: {len(cost_df_raw):,}")
print(f"  Columns: {cost_df_raw.columns.tolist()}")
if len(cost_df_raw):
    print(f"  Sample:")
    print(cost_df_raw.head(3).to_string())

In [ ]:
# ── Build state × fuel cost lookup ───────────────────────────────────────────

cost_df = cost_df_raw.copy()
cost_df['cost_per_mmbtu'] = pd.to_numeric(cost_df.get('cost-per-btu', pd.Series(dtype=float)),
                                           errors='coerce')
# Keep only 2-char state locations (exclude census region codes like '90')
cost_df = cost_df[cost_df['location'].str.len() == 2].copy()
cost_df = cost_df[cost_df['cost_per_mmbtu'].notna()].copy()

print(f"  State-level rows with valid cost: {len(cost_df):,}")
print(f"  Fuel type IDs present: {sorted(cost_df['fueltypeid'].unique().tolist())}")
print()

# ── Map EIA energy_source_code → epod fueltypeid ─────────────────────────────
# Mapping to the most specific matching fueltypeid in the epod data
ESC_TO_FUELTYPEID = {
    # Natural gas
    'NG' : 'NG',  'OG' : 'NG',  'BFG': 'NG',  'SGC': 'NG',  'PG' : 'NG',
    # Coal
    'BIT': 'BIT', 'SUB': 'SUB', 'LIG': 'LIG', 'RC' : 'RC',  'ANT': 'ANT',
    'WC' : 'WOC', 'SC' : 'SUB',
    # Petroleum
    'DFO': 'DFO', 'RFO': 'RFO', 'KER': 'PEL', 'JF' : 'PEL',
    'WO' : 'PEL', 'OO' : 'PEL', 'PC' : 'PC',
    # Nuclear — separate query if needed; typically ~0.6 $/MMBtu
    'NUC': 'NUC',
    # Biomass / waste
    'WDS': 'WAS', 'BLQ': 'WAS', 'OBS': 'WAS', 'AB' : 'WAS',
    'MSW': 'WAS', 'OBL': 'WAS', 'LFG': 'LFG',
    'OBG': 'BIO', 'OG2': 'BIO',
    # No fuel cost
    'WAT': None, 'WND': None, 'SUN': None, 'GEO': None,
    'ES' : None, 'PH' : None, 'FW' : None, 'WH' : None, 'PUR': None,
    'MWH': None,
}

# Build lookup: location(state) × fueltypeid → median cost
state_fuel_cost = (
    cost_df
    .groupby(['location', 'fueltypeid'])['cost_per_mmbtu']
    .median()
    .reset_index()
    .rename(columns={'location': 'stateid'})
)

# National fallback: median across all states per fuel type
national_fuel_cost = (
    cost_df
    .groupby('fueltypeid')['cost_per_mmbtu']
    .median()
    .reset_index()
    .rename(columns={'cost_per_mmbtu': 'national_cost'})
)

print(f"  State×fuel lookup entries: {len(state_fuel_cost):,}")
print(f"\n  National median fuel costs ($/MMBtu):")
print(national_fuel_cost.to_string())

# ── Join to generators ────────────────────────────────────────────────────────
included2['_fuel_key'] = included2['energy_source_code'].map(ESC_TO_FUELTYPEID)

# Primary join: state × fuel type
included3 = included2.merge(
    state_fuel_cost,
    left_on=['stateid', '_fuel_key'],
    right_on=['stateid', 'fueltypeid'],
    how='left',
).rename(columns={'cost_per_mmbtu': 'fuel_cost_per_mmbtu'})

# Fill missing with national median
missing_mask = included3['fuel_cost_per_mmbtu'].isna() & included3['_fuel_key'].notna()
if missing_mask.sum() > 0:
    nat_map = national_fuel_cost.set_index('fueltypeid')['national_cost'].to_dict()
    included3.loc[missing_mask, 'fuel_cost_per_mmbtu'] = (
        included3.loc[missing_mask, '_fuel_key'].map(nat_map)
    )
    included3['fuel_cost_imputed'] = missing_mask
else:
    included3['fuel_cost_imputed'] = False

# No fuel cost for non-thermal fuels
included3.loc[included3['_fuel_key'].isna(), 'fuel_cost_per_mmbtu'] = np.nan

n_cost = included3['fuel_cost_per_mmbtu'].notna().sum()
print(f"\n  Generators with fuel cost    : {n_cost:,}")
print(f"  Imputed (national median)    : {included3['fuel_cost_imputed'].sum():,}")

## Step 6 — NREL ATB 2024 O&M Costs

Source: `https://oedi-data-lake.s3.amazonaws.com/ATB/electricity/csv/2024/v3.0.0/ATBe.csv`
(v3.0.0, April 2025).  Moderate scenario, base year 2022 (values in 2022 USD).

**Nuclear O&M is not in ATB 2024.**  Hardcoded from NEI 2024 industry average:
FOM = 120 $/kW-yr, VOM = 2.0 $/MWh (https://www.nei.org/resources/statistics).

In [ ]:
print("─" * 60)
print("STEP 6 — Load NREL ATB 2024 O&M values")
print("─" * 60)

ATB_URL = "https://oedi-data-lake.s3.amazonaws.com/ATB/electricity/csv/2024/v3.0.0/ATBe.csv"

atb_raw = pd.read_csv(ATB_URL, low_memory=False)
print(f"  ATB rows: {len(atb_raw):,}")

# Filter to O&M parameters, Moderate scenario, 2022 base-year values
atb_om = atb_raw[
    atb_raw['core_metric_parameter'].isin(['Fixed O&M', 'Variable O&M']) &
    (atb_raw['scenario'] == 'Moderate') &
    (atb_raw['core_metric_variable'] == 2022)
].copy()

# Use default=1 rows where available; fall back to all rows
atb_om_default = atb_om[atb_om['default'] == 1].copy()
print(f"  ATB O&M Moderate 2022 (default=1): {len(atb_om_default):,} rows")
print(f"  Technologies: {sorted(atb_om_default['technology'].unique().tolist())}")

# Pivot to one row per technology: columns fom and vom
atb_pivot = (
    atb_om_default
    .groupby(['technology', 'technology_alias', 'core_metric_parameter'])['value']
    .mean()
    .unstack('core_metric_parameter')
    .reset_index()
)
atb_pivot.columns.name = None
atb_pivot = atb_pivot.rename(columns={
    'Fixed O&M'   : 'fom_per_kw_yr',
    'Variable O&M': 'vom_per_mwh',
})
if 'vom_per_mwh' not in atb_pivot.columns:
    atb_pivot['vom_per_mwh'] = 0.0
atb_pivot['vom_per_mwh'] = atb_pivot['vom_per_mwh'].fillna(0.0)

print(f"\n  ATB O&M pivot:")
print(atb_pivot[['technology', 'technology_alias', 'fom_per_kw_yr', 'vom_per_mwh']].to_string())

In [ ]:
# ── Supplement with explicit NGCC and NGCT from NaturalGas_FE (default=0) ────
# ATB 2024 default=1 for gas is NaturalGas_Retrofits (existing fleet CCS retrofit).
# For dispatch modeling of existing NGCC/NGCT fleet, use NaturalGas_FE
# F-Frame representative cases (CT and 2-on-1 CC).

ng_fe = atb_om[
    (atb_om['technology'] == 'NaturalGas_FE') &
    (atb_om['techdetail'].isin([
        'NG Combustion Turbine (F-Frame)',
        'NG 2-on-1 Combined Cycle (F-Frame)',
    ]))
].groupby(['techdetail', 'core_metric_parameter'])['value'].mean().unstack().reset_index()
ng_fe.columns.name = None
ng_fe = ng_fe.rename(columns={'Fixed O&M': 'fom_per_kw_yr', 'Variable O&M': 'vom_per_mwh'})
if 'vom_per_mwh' not in ng_fe.columns:
    ng_fe['vom_per_mwh'] = 0.0
ng_fe['vom_per_mwh'] = ng_fe['vom_per_mwh'].fillna(0.0)
ng_fe['technology'] = 'NaturalGas_FE'
print("  NaturalGas_FE CC and CT values:")
print(ng_fe.to_string())

ngct_fom = float(ng_fe.loc[ng_fe['techdetail'].str.contains('Combustion'), 'fom_per_kw_yr'].values[0])
ngct_vom = float(ng_fe.loc[ng_fe['techdetail'].str.contains('Combustion'), 'vom_per_mwh'].values[0])
ngcc_fom = float(ng_fe.loc[ng_fe['techdetail'].str.contains('Combined'),   'fom_per_kw_yr'].values[0])
ngcc_vom = float(ng_fe.loc[ng_fe['techdetail'].str.contains('Combined'),   'vom_per_mwh'].values[0])

print(f"\n  Using: NGCC FOM={ngcc_fom} $/kW-yr, VOM={ngcc_vom} $/MWh")
print(f"         NGCT FOM={ngct_fom} $/kW-yr, VOM={ngct_vom} $/MWh")

# Build the ATB lookup keyed to ATB technology name.
# Some technologies have multiple techdetail rows in default=1 → average their O&M values.
atb_lookup_df = (
    atb_pivot
    .groupby('technology')[['fom_per_kw_yr', 'vom_per_mwh']]
    .mean()
)
atb_lookup = atb_lookup_df.to_dict('index')

# Nuclear hardcoded (ATB 2024 omits nuclear O&M)
# Source: NEI 2024 industry average (https://www.nei.org/resources/statistics)
atb_lookup['Nuclear_hardcoded'] = {'fom_per_kw_yr': 120.0, 'vom_per_mwh': 2.0}

# NGCC / NGCT explicit entries
atb_lookup['NGCC'] = {'fom_per_kw_yr': ngcc_fom, 'vom_per_mwh': ngcc_vom}
atb_lookup['NGCT'] = {'fom_per_kw_yr': ngct_fom, 'vom_per_mwh': ngct_vom}

print(f"\n  ATB lookup table built: {len(atb_lookup)} entries")
print("  Keys:", sorted(atb_lookup.keys()))

In [ ]:
# ── Map EIA technology → ATB O&M lookup key ───────────────────────────────────
# Sources: ATB 2024 tech names + NGCC/NGCT from NaturalGas_FE F-Frame cases

EIA_TECH_TO_ATB = {
    # Natural gas
    'Natural Gas Fired Combined Cycle'        : 'NGCC',
    'Natural Gas Fired Combustion Turbine'    : 'NGCT',
    'Natural Gas Internal Combustion Engine'  : 'NGCT',
    'Natural Gas Steam Turbine'               : 'NaturalGas_Retrofits',
    'Natural Gas with Compressed Air Storage' : 'NaturalGas_Retrofits',
    'Other Natural Gas'                       : 'NaturalGas_Retrofits',
    # Coal
    'Conventional Steam Coal'                 : 'Coal_FE',
    'Coal Integrated Gasification Combined Cycle': 'Coal_FE',
    # Petroleum / other thermal
    'Petroleum Liquids'                       : 'NGCT',    # peaker proxy
    'Petroleum Coke'                          : 'Coal_FE',
    # Renewables
    'Onshore Wind Turbine'                    : 'LandbasedWind',
    'Offshore Wind Turbine'                   : 'OffShoreWind',
    'Solar Photovoltaic'                      : 'UtilityPV',
    'Solar Thermal with Energy Storage'       : 'CSP',
    'Solar Thermal without Energy Storage'    : 'CSP',
    # Hydro
    'Conventional Hydroelectric'              : 'Hydropower',
    'Hydroelectric Pumped Storage'            : 'Pumped Storage Hydropower',
    # Storage
    'Batteries'                               : 'Utility-Scale Battery Storage',
    'Flywheels'                               : 'Utility-Scale Battery Storage',
    # Other
    'Nuclear'                                 : 'Nuclear_hardcoded',
    'Geothermal'                              : 'Geothermal',
    'Landfill Gas'                            : 'Biopower',
    'Municipal Solid Waste'                   : 'Biopower',
    'Wood/Wood Waste Biomass'                 : 'Biopower',
    'Other Waste Biomass'                     : 'Biopower',
    'All Other'                               : 'NaturalGas_Retrofits',
    'Other Gases'                             : 'NaturalGas_Retrofits',
}

# Apply mapping
included3['_atb_key'] = included3['technology'].map(EIA_TECH_TO_ATB)
unmapped = included3[included3['_atb_key'].isna()]['technology'].unique()
if len(unmapped):
    print(f"  ⚠  Unmapped technologies (falling back to NaturalGas_Retrofits):")
    for t in sorted(unmapped): print(f"     {t}")
    included3['_atb_key'] = included3['_atb_key'].fillna('NaturalGas_Retrofits')

included3['fom_per_kw_yr'] = included3['_atb_key'].map(
    lambda k: atb_lookup.get(k, {}).get('fom_per_kw_yr', np.nan)
)
included3['vom_per_mwh'] = included3['_atb_key'].map(
    lambda k: atb_lookup.get(k, {}).get('vom_per_mwh', 0.0)
)

print(f"  Technologies with ATB O&M coverage: "
      f"{included3['fom_per_kw_yr'].notna().sum():,} / {len(included3):,}")
print()
print("  ATB key distribution (counts):")
print(included3['_atb_key'].value_counts().to_string())

## Step 7 — Compute Marginal Costs

- Thermal: `marginal_cost_per_mwh` = `heat_rate_mmbtu_mwh` × `fuel_cost_per_mmbtu` + `vom_per_mwh`
- Non-thermal: `marginal_cost_per_mwh` = `vom_per_mwh` only

In [ ]:
print("─" * 60)
print("STEP 7 — Compute marginal costs")
print("─" * 60)

df = included3.copy()

# Fuel component (only meaningful for thermal generators)
df['fuel_component'] = df['heat_rate_mmbtu_mwh'] * df['fuel_cost_per_mmbtu']

# Marginal cost
df['marginal_cost_per_mwh'] = np.where(
    df['heat_rate_mmbtu_mwh'].notna() & df['fuel_cost_per_mmbtu'].notna(),
    df['fuel_component'] + df['vom_per_mwh'],
    df['vom_per_mwh'],   # non-thermal: VOM only
)

# Sanity check: flag implausible values
hi = df['marginal_cost_per_mwh'] > 1000
lo = df['marginal_cost_per_mwh'] < 0
print(f"  Marginal cost > $1000/MWh : {hi.sum():,}  (capped at 1000 for output)")
print(f"  Marginal cost < $0/MWh   : {lo.sum():,}")
df.loc[hi, 'marginal_cost_per_mwh'] = 1000.0

print(f"\n  Marginal cost distribution (all generators):")
print(df['marginal_cost_per_mwh'].describe().to_string())
print()
print(f"  By technology group (mean $/MWh):")
tech_mc = (
    df.groupby('technology')
    .agg(n=('marginal_cost_per_mwh','count'),
         mean_mc=('marginal_cost_per_mwh','mean'),
         total_mw=('capacity_mw','sum'))
    .sort_values('total_mw', ascending=False)
)
print(tech_mc.to_string())

## Step 8 — Build Output DataFrame

In [ ]:
print("─" * 60)
print("STEP 8 — Build output DataFrame")
print("─" * 60)

OUTPUT_COLS = [
    'bus_id',
    'plantid',
    'generatorid',
    'technology',
    'energy-source-desc',       # fuel_type (human-readable)
    'energy_source_code',       # fuel type (EIA code)
    'capacity_mw',
    'heat_rate_mmbtu_mwh',      # nullable
    'fuel_cost_per_mmbtu',      # nullable for non-thermal
    'vom_per_mwh',
    'fom_per_kw_yr',
    'marginal_cost_per_mwh',
    'in_giant_component',
    'stateid',
    'plantName',
    'snap_dist_m',
    'fuel_cost_imputed',
]
# Only keep columns that exist
existing_cols = [c for c in OUTPUT_COLS if c in df.columns]
gen_costs = df[existing_cols].copy()

# Rename for cleaner parquet schema
gen_costs = gen_costs.rename(columns={
    'plantid'           : 'plant_id',
    'generatorid'       : 'generator_id',
    'energy-source-desc': 'fuel_type',
    'energy_source_code': 'fuel_code',
})

# Enforce column types
gen_costs['bus_id']                = gen_costs['bus_id'].astype(int)
gen_costs['capacity_mw']           = gen_costs['capacity_mw'].astype(float)
gen_costs['heat_rate_mmbtu_mwh']   = gen_costs['heat_rate_mmbtu_mwh'].astype(float)
gen_costs['fuel_cost_per_mmbtu']   = gen_costs['fuel_cost_per_mmbtu'].astype(float)
gen_costs['vom_per_mwh']           = gen_costs['vom_per_mwh'].astype(float)
gen_costs['fom_per_kw_yr']         = gen_costs['fom_per_kw_yr'].astype(float)
gen_costs['marginal_cost_per_mwh'] = gen_costs['marginal_cost_per_mwh'].astype(float)
gen_costs['in_giant_component']    = gen_costs['in_giant_component'].astype(bool)
gen_costs['snap_dist_m']           = gen_costs['snap_dist_m'].astype(float)
gen_costs['fuel_cost_imputed']     = gen_costs['fuel_cost_imputed'].astype(bool)

print(f"  Output shape: {gen_costs.shape}")
print(f"  Columns: {gen_costs.columns.tolist()}")
print()
print(gen_costs.head(3).to_string())

## Step 9 — Save Outputs

In [ ]:
print("─" * 60)
print("STEP 9 — Save outputs")
print("─" * 60)

# ── generators_with_costs.parquet ─────────────────────────────────────────────
parquet_path = PROCESSED / "generators_with_costs.parquet"
gen_costs.to_parquet(parquet_path, index=False)
print(f"  Saved generators_with_costs.parquet → {parquet_path}")
print(f"  Rows: {len(gen_costs):,}  |  Size: {parquet_path.stat().st_size/1024:.1f} KB")

# ── cost_coverage.csv ─────────────────────────────────────────────────────────
# One row per technology: count, MW, real vs imputed fractions, mean marginal cost
cov_rows = []
for tech, grp in gen_costs.groupby('technology'):
    n_total      = len(grp)
    total_mw     = grp['capacity_mw'].sum()
    # Heat rate: 'real' = came from EIA facility-fuel; non-thermal = N/A
    thermal_mask = grp['heat_rate_mmbtu_mwh'].notna() | grp['fuel_type'].str.lower().str.contains(
        'wind|solar|hydro|pump|batter|flywheel|geotherm', na=False)
    n_with_hr    = grp['heat_rate_mmbtu_mwh'].notna().sum()
    n_thermal_g  = (~grp['fuel_code'].isin(NO_HEAT_RATE_FUELS)).sum()
    hr_coverage  = n_with_hr / n_thermal_g if n_thermal_g > 0 else np.nan
    # Fuel cost
    n_with_fc    = grp['fuel_cost_per_mmbtu'].notna().sum()
    n_imputed_fc = grp['fuel_cost_imputed'].sum() if 'fuel_cost_imputed' in grp.columns else 0
    fc_coverage  = (n_with_fc - n_imputed_fc) / n_with_fc if n_with_fc > 0 else np.nan
    mean_mc      = grp['marginal_cost_per_mwh'].mean()
    cov_rows.append({
        'technology'          : tech,
        'n_generators'        : n_total,
        'total_mw'            : round(total_mw, 1),
        'frac_real_heat_rate' : round(float(hr_coverage), 4) if not np.isnan(float(hr_coverage if hr_coverage is not None else np.nan)) else np.nan,
        'frac_real_fuel_cost' : round(float(fc_coverage), 4) if not np.isnan(float(fc_coverage if fc_coverage is not None else np.nan)) else np.nan,
        'mean_marginal_cost_per_mwh': round(mean_mc, 2),
    })

coverage_df = pd.DataFrame(cov_rows).sort_values('total_mw', ascending=False)
csv_path    = PROCESSED / "cost_coverage.csv"
coverage_df.to_csv(csv_path, index=False)
print(f"  Saved cost_coverage.csv → {csv_path}")
print()
print(coverage_df.to_string())

## Step 10 — Coverage Summary

In [ ]:
print("─" * 60)
print("STEP 10 — Final coverage summary")
print("─" * 60)

total_mw_all = gen_costs['capacity_mw'].sum()

# Fraction of total MW with REAL (not imputed) heat rate
# 'real' = from EIA facility-fuel API; non-thermal = 'N/A' (not imputed)
thermal_gens  = gen_costs[~gen_costs['fuel_code'].isin(NO_HEAT_RATE_FUELS)]
hr_real_mw    = thermal_gens.loc[thermal_gens['heat_rate_mmbtu_mwh'].notna(), 'capacity_mw'].sum()
hr_total_mw   = thermal_gens['capacity_mw'].sum()

fc_real_mw    = gen_costs.loc[
    gen_costs['fuel_cost_per_mmbtu'].notna() & ~gen_costs['fuel_cost_imputed'],
    'capacity_mw'
].sum()
fc_thermal_mw = gen_costs.loc[gen_costs['fuel_cost_per_mmbtu'].notna(), 'capacity_mw'].sum()

print(f"  Total MW in output               : {total_mw_all:,.1f}")
print(f"  Thermal MW                       : {hr_total_mw:,.1f}")
print(f"  Thermal MW with real heat rate   : {hr_real_mw:,.1f}  ({100*hr_real_mw/hr_total_mw:.1f}%)")
print(f"  MW with real (state) fuel cost   : {fc_real_mw:,.1f}  ({100*fc_real_mw/fc_thermal_mw:.1f}% of thermal)")
print()

# Per-technology coverage breakdown
print(f"  Per-technology coverage (MW-weighted):")
print(f"  {'Technology':<45} {'Total MW':>9}  {'HR real%':>8}  {'FC real%':>8}  {'Mean MC':>8}")
print("  " + "-" * 86)

LOW_COVERAGE_FLAG = False
for _, row in coverage_df.iterrows():
    hr_pct = f"{100*row['frac_real_heat_rate']:.0f}%" if pd.notna(row['frac_real_heat_rate']) else '  N/A'
    fc_pct = f"{100*row['frac_real_fuel_cost']:.0f}%" if pd.notna(row['frac_real_fuel_cost']) else '  N/A'
    mc_str = f"${row['mean_marginal_cost_per_mwh']:>6.1f}"

    # Flag low coverage
    warn = ''
    if pd.notna(row['frac_real_heat_rate']) and row['frac_real_heat_rate'] < 0.5:
        warn += ' ⚠ hr'
        LOW_COVERAGE_FLAG = True
    if pd.notna(row['frac_real_fuel_cost']) and row['frac_real_fuel_cost'] < 0.5:
        warn += ' ⚠ fc'
        LOW_COVERAGE_FLAG = True

    print(f"  {row['technology']:<45} {row['total_mw']:>9,.0f}  {hr_pct:>8}  {fc_pct:>8}  {mc_str}{warn}")

print()
if LOW_COVERAGE_FLAG:
    print("  ⚠  One or more technologies have real data coverage < 50%.")
    print("     These rely heavily on imputed (state-median or ATB) values.")
    print("     Review cost_coverage.csv before using in dispatch optimization.")
else:
    print("  All technologies meet ≥50% real data coverage threshold.")

print()
print("  ── OUTPUTS ──")
print(f"  data/processed/network_metadata.json")
print(f"  data/processed/generators_with_costs.parquet")
print(f"  data/processed/cost_coverage.csv")